# Institutional Accumulation — **Balanced V4.7** (monthly, NSE)

Python port of Pine **Institutional Accumulation - Balanced V4.7** (no lookahead). Defaults match TradingView:

- Accumulation score ≥ 3 (relative volume, money flow sum, up-volume, OBV, ADL)
- Entry EMA slope + ADX + efficiency ratio + breakout filter
- ATR initial stop (tighter of 2.5× ATR or 20%), profit trail, larger-trend exit
- 100% of equity per long (Pine default), next **month open** fills — **each symbol backtested alone** (matches TV overlay)
- **Gap hygiene:** drop symbols with **≥5 missing sessions** in a row (`max_internal_gap_sessions=4`) + monthly-gap filter + `missing_monthly_bar` exits

Outputs: `$QUANT_DATA_DIR/results/institutional_accumulation_monthly/` · **per-symbol** (TV-style) backtest + `backtest/universe_trades.csv`

**Resume:** each phase writes parquet + `meta/*_meta.json` fingerprints. Skip rebuild unless `CFG["rerun_all"] = True` or `CFG["force_rerun"]` lists the phase (e.g. `"sync"`, `"daily"`).

**Phase 0** syncs Upstox daily candles into local cache + `nse_cm_*.parquet` (only missing symbol ranges and missing weekdays; ~364-day API chunks). Honours Upstox limits (defaults ~1 req/s, 2000/30min cap) — full universe may need several runs; cache is reused. Phase 1 builds the daily panel from those parquets.

In [ ]:
from pathlib import Path
import json, sys, subprocess, importlib.util, warnings
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "quant").is_dir():
    REPO_ROOT = REPO_ROOT.parent
for p in (REPO_ROOT / "src", REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

for pkg in ("pandas", "pyarrow", "numpy", "tqdm"):
    root = pkg.split(".")[0].replace("-", "_")
    if importlib.util.find_spec(root) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from quant.config import DATA_DIR, NIFTY50_PATH
from quant.research.explosive_research import (
    fingerprint_dict,
    force_rerun_phase,
    phase_cache_hit,
    write_phase_meta,
    read_meta_fingerprint,
    file_fingerprint,
)
from quant.research.institutional_accumulation import (
    DEFAULT_PARAMS,
    DEFAULT_SYMBOL_CONTINUITY,
    INST_ACC_ENGINE_VERSION,
    strategy_params_fingerprint,
    ensure_upstox_daily_for_monthly,
    load_daily_ohlcv_panel,
    filter_daily_panel_continuity,
    filter_monthly_panel_gaps,
    resample_panel_to_monthly,
    compute_signals_monthly,
    backtest_universe_monthly,
    backtest_symbol_monthly,
    summarize_backtest,
    summarize_universe,
)

from quant.config import PARQUET_DATA_DIR

OUT_DIR = DATA_DIR / "results" / "institutional_accumulation_monthly"
META_DIR = OUT_DIR / "meta"
for sub in tqdm(("panel", "monthly", "signals", "backtest", META_DIR), desc="Setup — output dirs"):
    (OUT_DIR / sub if sub != META_DIR else META_DIR).mkdir(parents=True, exist_ok=True)

DAILY_PANEL_PATH = OUT_DIR / "panel" / "daily_ohlcv_filtered.parquet"
MONTHLY_PATH = OUT_DIR / "monthly" / "ohlcv_monthly.parquet"
SIGNALS_PATH = OUT_DIR / "signals" / "monthly_signals.parquet"
TRADES_PATH = OUT_DIR / "backtest" / "universe_trades.parquet"
TRADES_CSV = OUT_DIR / "backtest" / "universe_trades.csv"
SYMBOL_STATS_PATH = OUT_DIR / "backtest" / "per_symbol_summary.parquet"
SYMBOL_STATS_CSV = OUT_DIR / "backtest" / "per_symbol_summary.csv"
CONTINUITY_EXCL_PATH = OUT_DIR / "reports" / "symbol_continuity_excluded.parquet"
MONTHLY_GAP_EXCL_PATH = OUT_DIR / "reports" / "monthly_gap_excluded.parquet"

SYNC_META = META_DIR / "sync_meta.json"
DAILY_META = META_DIR / "daily_meta.json"
MONTHLY_META = META_DIR / "monthly_meta.json"
SIGNALS_META = META_DIR / "signals_meta.json"
BACKTEST_META = META_DIR / "backtest_meta.json"

pd.set_option("display.width", 200)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)

## Config

In [ ]:
with tqdm(total=1, desc="Phase 0 — config") as pbar:
    CFG = {
        "rerun_all": False,
        "force_rerun": [],
        "date_start": "2005-01-01",
        "date_end": None,
        "spotlight_symbol": None,
        "sync_upstox": True,
        "sync_symbols": None,
        "sync_max_symbols": None,
        "sync_dry_run": False,
        "symbol_continuity": dict(DEFAULT_SYMBOL_CONTINUITY),
        **strategy_params_fingerprint(DEFAULT_PARAMS),
    }
    PARAMS = {k: CFG[k] for k in DEFAULT_PARAMS if k in CFG}
    pbar.update(1)

print("Engine:", INST_ACC_ENGINE_VERSION)
print("Strategy params:", {k: PARAMS[k] for k in (
    "accumulation_needed", "initial_stop_atr", "maximum_initial_stop_percent",
    "cooldown_bars", "position_pct", "initial_capital",
)})

## Phase 0 — Upstox daily sync (incremental, local cache)

In [ ]:
from quant.data.tejhq import latest_local_parquet_date

_sync_fp = fingerprint_dict({
    "engine": INST_ACC_ENGINE_VERSION,
    "date_start": CFG["date_start"],
    "date_end": CFG["date_end"],
    "sync_symbols": CFG.get("sync_symbols"),
    "sync_max_symbols": CFG.get("sync_max_symbols"),
    "parquet_latest": str(latest_local_parquet_date(PARQUET_DATA_DIR)),
})

sync_summary = None
if CFG.get("sync_upstox") and not force_rerun_phase(CFG, "sync") and SYNC_META.is_file():
  try:
    prev = json.loads(SYNC_META.read_text())
    if prev.get("fingerprint") == _sync_fp:
        sync_summary = prev.get("summary")
  except Exception:
    pass

if sync_summary is None and CFG.get("sync_upstox"):
    with tqdm(total=1, desc="Phase 0 — Upstox sync") as bar:
        sync_summary = ensure_upstox_daily_for_monthly(
            date_start=CFG["date_start"],
            date_end=CFG["date_end"],
            symbols=CFG.get("sync_symbols"),
            max_symbols=CFG.get("sync_max_symbols"),
            dry_run=bool(CFG.get("sync_dry_run")),
        )
        bar.update(1)
    write_phase_meta(SYNC_META, "sync", _sync_fp, summary=sync_summary)
elif not CFG.get("sync_upstox"):
    print("Phase 0 skipped (sync_upstox=False)")

if sync_summary:
    print("Upstox sync:", sync_summary)
print("Latest nse_cm parquet:", latest_local_parquet_date(PARQUET_DATA_DIR))

## Phase 1 — daily panel (cached)

In [ ]:
_daily_fp = fingerprint_dict({
    "engine": INST_ACC_ENGINE_VERSION,
    "date_start": CFG["date_start"],
    "date_end": CFG["date_end"],
    "eq_only": PARAMS["eq_only"],
    "symbol_continuity": CFG.get("symbol_continuity"),
    "sync": read_meta_fingerprint(SYNC_META) if SYNC_META.is_file() else None,
    "parquet_latest": str(latest_local_parquet_date(PARQUET_DATA_DIR)),
})

if phase_cache_hit(CFG, "daily", [DAILY_PANEL_PATH], DAILY_META, _daily_fp):
    daily = pd.read_parquet(DAILY_PANEL_PATH)
else:
    with tqdm(total=4, desc="Phase 1 — daily panel") as bar:
        raw = load_daily_ohlcv_panel(
            date_start=CFG["date_start"],
            date_end=CFG["date_end"],
            eq_only=PARAMS["eq_only"],
            from_nse_cm=True,
            parquet_root=PARQUET_DATA_DIR,
        )
        bar.update(1)
        nifty = pd.read_parquet(NIFTY50_PATH) if NIFTY50_PATH.exists() else None
        if nifty is not None:
            nifty = nifty.rename(columns={c: c.title() for c in nifty.columns})
        daily, continuity_excl = filter_daily_panel_continuity(
            raw, CFG.get("symbol_continuity"), nifty_df=nifty,
        )
        bar.update(1)
        CONTINUITY_EXCL_PATH.parent.mkdir(parents=True, exist_ok=True)
        continuity_excl.to_parquet(CONTINUITY_EXCL_PATH, index=False)
        bar.update(1)
        daily.to_parquet(DAILY_PANEL_PATH, index=False)
        write_phase_meta(
            DAILY_META,
            "daily",
            _daily_fp,
            n_rows=len(daily),
            n_excluded_continuity=int(len(continuity_excl)),
        )
        bar.update(1)

daily["Date"] = pd.to_datetime(daily["Date"])
n_excl = len(pd.read_parquet(CONTINUITY_EXCL_PATH)) if CONTINUITY_EXCL_PATH.exists() else 0
print("Daily:", daily.shape, daily["Date"].min(), "→", daily["Date"].max(), "| symbols:", daily["Symbol"].nunique())
print("Excluded (session gaps):", n_excl, CONTINUITY_EXCL_PATH)

## Phase 2 — resample to calendar months (cached)

In [ ]:
_monthly_fp = fingerprint_dict({
    "daily": read_meta_fingerprint(DAILY_META),
    "engine": INST_ACC_ENGINE_VERSION,
    "max_monthly_gap_months": PARAMS["max_monthly_gap_months"],
})

if phase_cache_hit(CFG, "monthly", [MONTHLY_PATH], MONTHLY_META, _monthly_fp):
    monthly = pd.read_parquet(MONTHLY_PATH)
else:
    parts = []
    syms = daily["Symbol"].unique()
    for sym in tqdm(syms, desc="Phase 2 — monthly bars", unit="symbol"):
        g = daily.loc[daily["Symbol"] == sym, ["Date", "Open", "High", "Low", "Close", "Volume"]]
        from quant.research.institutional_accumulation import resample_symbol_to_monthly
        m = resample_symbol_to_monthly(g)
        if m.empty:
            continue
        m["Symbol"] = sym
        parts.append(m)
    monthly_all = pd.concat(parts, ignore_index=True)
    monthly, monthly_excl = filter_monthly_panel_gaps(
        monthly_all, int(PARAMS["max_monthly_gap_months"]),
    )
    MONTHLY_GAP_EXCL_PATH.parent.mkdir(parents=True, exist_ok=True)
    monthly_excl.to_parquet(MONTHLY_GAP_EXCL_PATH, index=False)
    monthly.to_parquet(MONTHLY_PATH, index=False)
    write_phase_meta(
        MONTHLY_META,
        "monthly",
        _monthly_fp,
        n_symbols=monthly["Symbol"].nunique(),
        n_excluded_monthly_gap=int(len(monthly_excl)),
    )

monthly["Date"] = pd.to_datetime(monthly["Date"])
n_mg = len(pd.read_parquet(MONTHLY_GAP_EXCL_PATH)) if MONTHLY_GAP_EXCL_PATH.exists() else 0
print("Monthly:", monthly.shape, "| symbols:", monthly["Symbol"].nunique())
print("Excluded (monthly gaps):", n_mg, MONTHLY_GAP_EXCL_PATH)

## Phase 3 — accumulation signals (cached)

In [ ]:
_signals_fp = fingerprint_dict({
    "monthly": read_meta_fingerprint(MONTHLY_META),
    "params": PARAMS,
    "engine": INST_ACC_ENGINE_VERSION,
})

if phase_cache_hit(CFG, "signals", [SIGNALS_PATH], SIGNALS_META, _signals_fp):
    signals = pd.read_parquet(SIGNALS_PATH)
else:
    signals = compute_signals_monthly(monthly, PARAMS)
    signals.to_parquet(SIGNALS_PATH, index=False)
    n_buy = int(signals["buy_signal"].sum()) if not signals.empty else 0
    write_phase_meta(SIGNALS_META, "signals", _signals_fp, n_buy=n_buy)

if signals.empty:
    raise RuntimeError("No monthly signals — check data range / min_monthly_bars.")
print("Signal rows:", len(signals), "| buy flags:", int(signals["buy_signal"].sum()))
display(signals.loc[signals["buy_signal"], [
    "Date", "Symbol", "accumulation_score", "relative_volume", "adx", "efficiency_ratio",
]].head(15))

## Phase 4 — per-symbol backtest, all valid stocks (cached)

In [ ]:
_bt_fp = fingerprint_dict({
    "signals": read_meta_fingerprint(SIGNALS_META),
    "params": PARAMS,
    "engine": INST_ACC_ENGINE_VERSION,
})

if phase_cache_hit(CFG, "backtest", [TRADES_PATH, SYMBOL_STATS_PATH], BACKTEST_META, _bt_fp):
    trades = pd.read_parquet(TRADES_PATH)
    symbol_stats = pd.read_parquet(SYMBOL_STATS_PATH)
else:
    trades, symbol_stats = backtest_universe_monthly(monthly, PARAMS)
    trades.to_parquet(TRADES_PATH, index=False)
    trades.to_csv(TRADES_CSV, index=False)
    symbol_stats.to_parquet(SYMBOL_STATS_PATH, index=False)
    symbol_stats.to_csv(SYMBOL_STATS_CSV, index=False)
    write_phase_meta(
        BACKTEST_META,
        "backtest",
        _bt_fp,
        n_symbols=len(symbol_stats),
        n_trades=len(trades),
    )

universe_summary = summarize_universe(symbol_stats)
print("Universe summary:", universe_summary)
print(
    "Strategy works on majority of stocks with trades:",
    universe_summary["majority_profitable_among_traders"],
)

## Phase 5 — universe report (fast; always runs)

In [ ]:
display(pd.DataFrame([universe_summary]))

cols = [
    "Symbol", "profitable", "had_trades", "total_return_pct", "cagr_pct",
    "max_drawdown_pct", "n_trades", "win_rate_pct", "n_monthly_bars",
]
cols = [c for c in cols if c in symbol_stats.columns]
traded = symbol_stats.loc[symbol_stats["had_trades"]]
print(f"Tested {len(symbol_stats)} symbols | {len(traded)} with ≥1 round-trip")
display(traded.sort_values("total_return_pct", ascending=False)[cols].head(25))
display(traded.sort_values("total_return_pct", ascending=True)[cols].head(15))

spot = CFG.get("spotlight_symbol")
if spot:
    row = symbol_stats.loc[symbol_stats["Symbol"] == spot.upper()]
    if not row.empty:
        display(row[cols])
        display(trades.loc[trades["Symbol"] == spot.upper()].tail(20))

trades.to_csv(TRADES_CSV, index=False)
symbol_stats.to_csv(SYMBOL_STATS_CSV, index=False)
print("Wrote", TRADES_CSV, "|", SYMBOL_STATS_CSV)

summary_path = OUT_DIR / "backtest" / "universe_summary.json"
summary_path.write_text(json.dumps(universe_summary, indent=2))
print("Wrote", summary_path)